# PSID Generic Crisis Module - Final Ranked Workflow

## Objective

This notebook now documents the production-ready, CSV-driven workflow for the PSID crisis module. Instead of using the older synthetic demonstration path, it works directly from the final ranked question file and the maintained artifact generator.

The workflow in this notebook does four things:

1. Loads the authoritative ranked dataset from `PSID_Ranked_Questions_Final.csv`.
2. Summarizes the selected 28-question module, including source, toggle, timing, and construct coverage.
3. Reviews the highest-ranked questions and the current stakeholder-facing deliverables.
4. Regenerates the dashboard bundle, report figures, and summary artifacts through the same helper used by the final HTML and markdown outputs.

## Production Pipeline

```
PSID_Ranked_Questions_Final.csv
    -> load_dataset() from generate_psid_artifacts.py
    -> selected module review and validation
    -> deliverable inventory and figure checks
    -> build_all() refresh for dashboard, report figures, and summary bundles
```

**Version**: Final ranked workflow
**Last Updated**: 2026-03-10

In [4]:
import importlib
import json
import pandas as pd
from IPython.display import display

import generate_psid_artifacts as artifacts

artifacts = importlib.reload(artifacts)

ROOT = artifacts.ROOT
CSV_PATH = artifacts.CSV_PATH
SUMMARY_PATH = artifacts.SUMMARY_PATH
DASHBOARD_DATA_PATH = artifacts.DASHBOARD_DATA_PATH
FIG_TOP = artifacts.FIG_TOP
FIG_TOGGLE = artifacts.FIG_TOGGLE
FIG_UTILITY = artifacts.FIG_UTILITY
FIG_HEATMAP = artifacts.FIG_HEATMAP
FIG_TIME = artifacts.FIG_TIME
build_all = artifacts.build_all
load_dataset = artifacts.load_dataset
build_summary = artifacts.build_summary
write_dashboard_data = artifacts.write_dashboard_data
save_aliases = artifacts.save_aliases
plot_top_ranked = artifacts.plot_top_ranked
plot_toggle_comparison = artifacts.plot_toggle_comparison
plot_utility_vs_burden = artifacts.plot_utility_vs_burden
plot_construct_heatmap = artifacts.plot_construct_heatmap
plot_time_budget = artifacts.plot_time_budget
_style_matplotlib = artifacts._style_matplotlib
LEGACY_ALIASES = artifacts.LEGACY_ALIASES

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_columns", 20)

print("Environment ready for the final ranked workflow")
print(f"Project root: {ROOT}")
print(f"Ranked CSV: {CSV_PATH.name}")
print(f"Summary bundle: {SUMMARY_PATH.name}")

Environment ready for the final ranked workflow
Project root: /Users/namomac/Team-PSID
Ranked CSV: PSID_Ranked_Questions_Final.csv
Summary bundle: psid_artifact_summary.json


In [5]:
df = load_dataset()
selected_df = df[df["selected"]].copy()

summary = {
    "rows": int(len(df)),
    "selected_rows": int(len(selected_df)),
    "selected_minutes": round(float(selected_df["minutes"].sum()), 2),
    "all_minutes": round(float(df["minutes"].sum()), 2),
    "avg_ri": round(float(df["Ri"].mean()), 3),
    "max_ri": round(float(df["Ri"].max()), 3),
}

summary

{'rows': 52,
 'selected_rows': 28,
 'selected_minutes': 29.17,
 'all_minutes': 68.95,
 'avg_ri': 2.079,
 'max_ri': 5.667}

In [6]:
preview_columns = [
    "question_text",
    "source",
    "module_type",
    "toggle_category",
    "Ui",
    "Bi",
    "Ri",
    "selected",
    "word_count",
    "minutes",
    "constructs",
]

display(df[preview_columns].head(12))

,question_text,source,module_type,toggle_category,Ui,Bi,Ri,selected,word_count,minutes,constructs
0,Any financial difficulties,COVID-19,Crisis-Specific,Generic Core,1.70,0.3,5.6667,True,3,0.350000,[Financial Coping]
1,Lost earnings because of the pandemic,COVID-19,Crisis-Specific,Toggle: Pandemic / Disaster,3.40,0.6,5.6667,True,6,0.700000,"[Economic / Income, Trauma / Health]"
2,stopped this work?,Govt Shutdown Income,Generic,Generic Core,1.40,0.3,4.6667,True,3,0.350000,[Employment]
3,Received stimulus payment,COVID-19,Crisis-Specific,Toggle: Pandemic / Disaster,1.40,0.3,4.6667,True,3,0.350000,[Government Aid]
4,stopped working at this business?,Govt Shutdown Income,Generic,Generic Core,2.25,0.5,4.5000,True,5,0.583333,"[Economic / Income, Employment]"
5,Working in a job that was considered essential work?,COVID-19,Crisis-Specific,Toggle: Pandemic / Disaster,3.60,0.9,4.0000,True,9,1.050000,[Employment]
6,How did you/your family manage any financial difficulties due to the shutdown - sell your belongings?,Govt Shutdown Crisis,Crisis-Specific,Toggle: Financial Crisis,5.95,1.6,3.7188,True,16,1.866667,"[Financial Coping, Government Aid]"
7,Only work from home,COVID-19,Crisis-Specific,Toggle: Pandemic / Disaster,1.40,0.4,3.5000,True,4,0.466667,[Employment]
8,Stimulus payments,COVID-19,Crisis-Specific,Toggle: Pandemic / Disaster,0.70,0.2,3.5000,True,2,0.233333,[Government Aid]
9,Paycheck protection,COVID-19,Crisis-Specific,Toggle: Pandemic / Disaster,0.70,0.2,3.5000,True,2,0.233333,[Government Aid]


## 2. Final Corpus Snapshot

In [7]:
source_counts = df["source"].value_counts().rename_axis("source").reset_index(name="questions")
toggle_counts = df["toggle_category"].value_counts().rename_axis("toggle_category").reset_index(name="questions")

print("Source distribution")
display(source_counts)

print("Toggle distribution")
display(toggle_counts)

print("Selected module timing by toggle")
display(
    selected_df.groupby("toggle_category", as_index=False)["minutes"]
    .sum()
    .sort_values("minutes", ascending=False)
    .rename(columns={"minutes": "selected_minutes"})
)

Source distribution


,source,questions
0,Hurricane Katrina 2007,37
1,COVID-19,8
2,Govt Shutdown Income,3
3,Understanding Society,3
4,Govt Shutdown Crisis,1


Toggle distribution


,toggle_category,questions
0,Toggle: Pandemic / Disaster,44
1,Generic Core,7
2,Toggle: Financial Crisis,1


Selected module timing by toggle


,toggle_category,selected_minutes
2,Toggle: Pandemic / Disaster,22.400000
0,Generic Core,4.900000
1,Toggle: Financial Crisis,1.866667


The ranked corpus contains 52 questions across five historical sources. The `selected` flag in the loaded dataframe is derived directly from `selected_for_module`, so every table below reflects the final 28-question module rather than a synthetic example.

## 3. Top Ranked Question Review

In [8]:
top_ranked = (
    df.sort_values(["Ri", "Ui"], ascending=[False, False])
    [["question_text", "source", "toggle_category", "Ui", "Bi", "Ri", "selected"]]
    .head(15)
    .reset_index(drop=True)
)

display(top_ranked)

,question_text,source,toggle_category,Ui,Bi,Ri,selected
0,Lost earnings because of the pandemic,COVID-19,Toggle: Pandemic / Disaster,3.40,0.6,5.6667,True
1,Any financial difficulties,COVID-19,Generic Core,1.70,0.3,5.6667,True
2,stopped this work?,Govt Shutdown Income,Generic Core,1.40,0.3,4.6667,True
3,Received stimulus payment,COVID-19,Toggle: Pandemic / Disaster,1.40,0.3,4.6667,True
4,stopped working at this business?,Govt Shutdown Income,Generic Core,2.25,0.5,4.5000,True
5,Working in a job that was considered essential work?,COVID-19,Toggle: Pandemic / Disaster,3.60,0.9,4.0000,True
6,How did you/your family manage any financial difficulties due to the shutdown - sell your belongings?,Govt Shutdown Crisis,Toggle: Financial Crisis,5.95,1.6,3.7188,True
7,Only work from home,COVID-19,Toggle: Pandemic / Disaster,1.40,0.4,3.5000,True
8,Stimulus payments,COVID-19,Toggle: Pandemic / Disaster,0.70,0.2,3.5000,True
9,Paycheck protection,COVID-19,Toggle: Pandemic / Disaster,0.70,0.2,3.5000,True


The highest-ranked items remain short economic hardship and work-disruption questions, with Katrina and COVID items providing the strongest crisis-specific additions once the generic core is in place.

## 4. Selected Module Composition

In [9]:
selected_breakdown = (
    selected_df.groupby(["toggle_category", "source"], as_index=False)
    .agg(
        selected_questions=("question_text", "count"),
        total_minutes=("minutes", "sum"),
        avg_ri=("Ri", "mean"),
    )
    .sort_values(["toggle_category", "total_minutes"], ascending=[True, False])
)

selected_breakdown["total_minutes"] = selected_breakdown["total_minutes"].round(2)
selected_breakdown["avg_ri"] = selected_breakdown["avg_ri"].round(3)

display(selected_breakdown)

,toggle_category,source,selected_questions,total_minutes,avg_ri
2,Generic Core,Understanding Society,3,2.45,1.354
1,Generic Core,Govt Shutdown Income,3,2.10,3.965
0,Generic Core,COVID-19,1,0.35,5.667
3,Toggle: Financial Crisis,Govt Shutdown Crisis,1,1.87,3.719
5,Toggle: Pandemic / Disaster,Hurricane Katrina 2007,13,18.43,2.100
4,Toggle: Pandemic / Disaster,COVID-19,7,3.97,3.879


The selected module remains within the 30-minute cap. The generic core contributes the stable always-on backbone, while the Pandemic / Disaster bank carries most of the detailed time burden after Katrina integration.

## 5. Construct Coverage in the Selected Module

In [10]:
construct_counts = (
    selected_df["constructs"]
    .explode()
    .dropna()
    .value_counts()
    .rename_axis("construct")
    .reset_index(name="selected_questions")
)

display(construct_counts)

,construct,selected_questions
0,Trauma / Health,13
1,Employment,7
2,Government Aid,5
3,Housing / Shelter,5
4,Economic / Income,4
5,Financial Coping,3
6,Demographics,3


Trauma / Health, Employment, Government Aid, and Housing / Shelter dominate the selected construct profile. That confirms the final module is not just an economic short form anymore; it is now materially shaped by displacement and trauma content.

## 6. Stakeholder Deliverables

In [20]:
deliverables = pd.DataFrame(
    [
        {"deliverable": "Dashboard", "file": "PSID_Crisis_Module_Dashboard.html"},
        {"deliverable": "Master questionnaire viewer", "file": "PSID_Module_Questionnaire_Master.html"},
        {"deliverable": "Generic questionnaire demo", "file": "PSID_Generic_Module_Questionnaire.html"},
        {"deliverable": "Crisis-specific questionnaire demo", "file": "PSID_Crisis_Specific_Questionnaire_Demo.html"},
        {"deliverable": "Final web report", "file": "PSID_NLP_Optimization_Report_Final.html"},
        {"deliverable": "Report source note", "file": "PSID_NLP_Optimization_Report_Final.docx.md"},
        {"deliverable": "Mirror source note", "file": "FinalReport.md"},
        {"deliverable": "Legacy report alias", "file": "FinalReport.html"},
        {"deliverable": "Summary JSON", "file": SUMMARY_PATH.name},
        {"deliverable": "Dashboard data bundle", "file": DASHBOARD_DATA_PATH.name},
    ]
)

deliverables["exists"] = deliverables["file"].apply(lambda path: (ROOT / path).exists())
display(deliverables)

,deliverable,file,exists
0,Dashboard,PSID_Crisis_Module_Dashboard.html,True
1,Master questionnaire viewer,PSID_Module_Questionnaire_Master.html,True
2,Generic questionnaire demo,PSID_Generic_Module_Questionnaire.html,True
3,Crisis-specific questionnaire demo,PSID_Crisis_Specific_Questionnaire_Demo.html,True
4,Final web report,PSID_NLP_Optimization_Report_Final.html,True
5,Report source note,PSID_NLP_Optimization_Report_Final.docx.md,True
6,Mirror source note,FinalReport.md,True
7,Legacy report alias,FinalReport.html,True
8,Summary JSON,psid_artifact_summary.json,True
9,Dashboard data bundle,psid_dashboard_data.js,True


The notebook now points directly to the same HTML, markdown source-note, JSON, and JavaScript outputs used outside Jupyter. The master questionnaire viewer remains the single entry point for reviewing both questionnaire demos together, while the final public report is now the dedicated HTML page.

## 7. Figure Outputs and Supporting Bundles

In [12]:
figure_paths = [FIG_TOP, FIG_TOGGLE, FIG_UTILITY, FIG_HEATMAP, FIG_TIME]
figure_status = pd.DataFrame(
    [
        {
            "figure": path.name,
            "exists": path.exists(),
            "size_kb": round(path.stat().st_size / 1024, 1) if path.exists() else None,
        }
        for path in figure_paths
    ]
)

display(figure_status)

artifact_summary = json.loads(SUMMARY_PATH.read_text())
display(pd.DataFrame([artifact_summary]))

,figure,exists,size_kb
0,fig_top_ranked_questions.png,True,162.4
1,fig_toggle_comparison.png,True,80.0
2,fig_utility_vs_burden.png,True,106.1
3,fig_construct_heatmap.png,True,98.8
4,fig_time_budget.png,True,57.5


,rows,selected_rows,selected_minutes,all_minutes,toggle_counts,selected_toggle_counts,source_counts,selected_source_counts,construct_counts,avg_ri,max_ri
0,52,28,29.17,68.95,"{'Toggle: Pandemic / Disaster': 44, 'Generic Core': 7, 'Toggle: Financial Crisis': 1}","{'Toggle: Pandemic / Disaster': 20, 'Generic Core': 7, 'Toggle: Financial Crisis': 1}","{'Hurricane Katrina 2007': 37, 'COVID-19': 8, 'Govt Shutdown Income': 3, 'Understanding Society': 3, 'Govt Shutdown Crisis': 1}","{'Hurricane Katrina 2007': 13, 'COVID-19': 8, 'Govt Shutdown Income': 3, 'Understanding Society': 3, 'Govt Shutdown Crisis': 1}","{'Trauma / Health': 13, 'Employment': 7, 'Government Aid': 5, 'Housing / Shelter': 5, 'Economic / Income': 4, 'Financial Coping': 3, 'De...",2.079,5.667


The figure files and browser-ready bundles are generated from the same helper script, so the dashboard, reports, and notebook refresh step stay synchronized as long as `build_all()` remains the only supported regeneration path.

## 8. Selected Question Inspection

In [13]:
selected_df[
    [
        "question_text",
        "source",
        "toggle_category",
        "Ri",
        "Ui",
        "Bi",
        "minutes",
    ]
]\
    .sort_values(["toggle_category", "Ri"], ascending=[True, False])\
    .reset_index(drop=True)

,question_text,source,toggle_category,Ri,Ui,Bi,minutes
0,Any financial difficulties,COVID-19,Generic Core,5.6667,1.70,0.3,0.350000
1,stopped this work?,Govt Shutdown Income,Generic Core,4.6667,1.40,0.3,0.350000
2,stopped working at this business?,Govt Shutdown Income,Generic Core,4.5000,2.25,0.5,0.583333
3,Were/Was there any wages or salarys from this job/these jobs?,Govt Shutdown Income,Generic Core,2.7273,3.00,1.1,1.166667
4,Calculate respondents age,Understanding Society,Generic Core,1.6667,0.50,0.3,0.350000
5,And are you... 1. Male 2. Female,Understanding Society,Generic Core,1.2273,1.35,1.1,0.816667
6,"Can I just check, are you normally resident at this address?",Understanding Society,Generic Core,1.1667,1.40,1.2,1.283333
7,How did you/your family manage any financial difficulties due to the shutdown - sell your belongings?,Govt Shutdown Crisis,Toggle: Financial Crisis,3.7188,5.95,1.6,1.866667
8,Lost earnings because of the pandemic,COVID-19,Toggle: Pandemic / Disaster,5.6667,3.40,0.6,0.700000
9,Received stimulus payment,COVID-19,Toggle: Pandemic / Disaster,4.6667,1.40,0.3,0.350000


In [14]:
selected_df.nlargest(10, "Ri")[
    ["question_text", "source", "toggle_category", "Ri", "minutes"]
]\
    .reset_index(drop=True)

,question_text,source,toggle_category,Ri,minutes
0,Any financial difficulties,COVID-19,Generic Core,5.6667,0.350000
1,Lost earnings because of the pandemic,COVID-19,Toggle: Pandemic / Disaster,5.6667,0.700000
2,stopped this work?,Govt Shutdown Income,Generic Core,4.6667,0.350000
3,Received stimulus payment,COVID-19,Toggle: Pandemic / Disaster,4.6667,0.350000
4,stopped working at this business?,Govt Shutdown Income,Generic Core,4.5000,0.583333
5,Working in a job that was considered essential work?,COVID-19,Toggle: Pandemic / Disaster,4.0000,1.050000
6,How did you/your family manage any financial difficulties due to the shutdown - sell your belongings?,Govt Shutdown Crisis,Toggle: Financial Crisis,3.7188,1.866667
7,Only work from home,COVID-19,Toggle: Pandemic / Disaster,3.5000,0.466667
8,Stimulus payments,COVID-19,Toggle: Pandemic / Disaster,3.5000,0.233333
9,Paycheck protection,COVID-19,Toggle: Pandemic / Disaster,3.5000,0.233333


In [15]:
pd.crosstab(
    selected_df["toggle_category"],
    selected_df["source"],
    margins=True,
    margins_name="total",
)

source,COVID-19,Govt Shutdown Crisis,Govt Shutdown Income,Hurricane Katrina 2007,Understanding Society,total
toggle_category,,,,,,
Generic Core,1,0,3,0,3,7
Toggle: Financial Crisis,0,1,0,0,0,1
Toggle: Pandemic / Disaster,7,0,0,13,0,20
total,8,1,3,13,3,28


In [16]:
selected_constructs = (
    selected_df[["toggle_category", "constructs"]]
    .explode("constructs")
    .dropna()
    .assign(construct=lambda frame: frame["constructs"].astype(str).str.strip())
)

pd.crosstab(
    selected_constructs["construct"],
    selected_constructs["toggle_category"],
    margins=True,
    margins_name="total",
)\
    .sort_values("total", ascending=False)\
    .head(15)

toggle_category,Generic Core,Toggle: Financial Crisis,Toggle: Pandemic / Disaster,total
construct,,,,
total,9,2,29,40
Trauma / Health,0,0,13,13
Employment,3,0,4,7
Government Aid,0,1,4,5
Housing / Shelter,0,0,5,5
Economic / Income,2,0,2,4
Demographics,3,0,0,3
Financial Coping,1,1,1,3


## 9. Validation Checks

These benchmarks are pinned to the current final ranked CSV so the notebook can flag accidental changes before the downstream HTML, figure, and markdown artifacts are refreshed.

In [17]:
validation_checks = pd.DataFrame(
    [
        {"check": "Rows in final corpus", "value": int(len(df)), "target": 52},
        {"check": "Rows selected for module", "value": int(selected_df.shape[0]), "target": 28},
        {"check": "Selected minutes", "value": round(float(selected_df["minutes"].sum()), 2), "target": 29.17},
        {"check": "Total corpus minutes", "value": round(float(df["minutes"].sum()), 2), "target": 68.95},
        {"check": "Average Ri", "value": round(float(df["Ri"].mean()), 3), "target": 2.079},
        {"check": "Maximum Ri", "value": round(float(df["Ri"].max()), 3), "target": 5.667},
    ]
)
validation_checks["pass"] = validation_checks["value"] == validation_checks["target"]
display(validation_checks)

,check,value,target,pass
0,Rows in final corpus,52.000,52.000,True
1,Rows selected for module,28.000,28.000,True
2,Selected minutes,29.170,29.170,True
3,Total corpus minutes,68.950,68.950,True
4,Average Ri,2.079,2.079,True
5,Maximum Ri,5.667,5.667,True


## 10. Refresh the Production Artifacts

Running `build_all()` is the supported end-to-end refresh path. It reloads the final CSV, regenerates the summary and dashboard bundles, refreshes the figures, and keeps the notebook aligned with the stakeholder-facing HTML deliverables.

In [18]:
updated_summary = build_all()
display(pd.DataFrame([updated_summary]))

,rows,selected_rows,selected_minutes,all_minutes,toggle_counts,selected_toggle_counts,source_counts,selected_source_counts,construct_counts,avg_ri,max_ri
0,52,28,29.17,68.95,"{'Toggle: Pandemic / Disaster': 44, 'Generic Core': 7, 'Toggle: Financial Crisis': 1}","{'Toggle: Pandemic / Disaster': 20, 'Generic Core': 7, 'Toggle: Financial Crisis': 1}","{'Hurricane Katrina 2007': 37, 'COVID-19': 8, 'Govt Shutdown Income': 3, 'Understanding Society': 3, 'Govt Shutdown Crisis': 1}","{'Hurricane Katrina 2007': 13, 'COVID-19': 8, 'Govt Shutdown Income': 3, 'Understanding Society': 3, 'Govt Shutdown Crisis': 1}","{'Trauma / Health': 13, 'Employment': 7, 'Government Aid': 5, 'Housing / Shelter': 5, 'Economic / Income': 4, 'Financial Coping': 3, 'De...",2.079,5.667


## 11. Refresh Checklist

After `build_all()` completes, confirm that the dashboard, questionnaire demos, master viewer, figures, and markdown reports still reflect the same ranked CSV and benchmark values shown earlier in this notebook. This notebook no longer maintains a separate experimental dashboard or CSV export path; it documents and verifies the final production artifacts only.

## 12. Post-Refresh Verification

This final check reloads the written summary bundle after `build_all()` so the notebook can verify the persisted artifact metrics, not just the in-memory dataframe state.

In [19]:
artifact_summary = json.loads(SUMMARY_PATH.read_text())
display(pd.DataFrame([artifact_summary]))

,rows,selected_rows,selected_minutes,all_minutes,toggle_counts,selected_toggle_counts,source_counts,selected_source_counts,construct_counts,avg_ri,max_ri
0,52,28,29.17,68.95,"{'Toggle: Pandemic / Disaster': 44, 'Generic Core': 7, 'Toggle: Financial Crisis': 1}","{'Toggle: Pandemic / Disaster': 20, 'Generic Core': 7, 'Toggle: Financial Crisis': 1}","{'Hurricane Katrina 2007': 37, 'COVID-19': 8, 'Govt Shutdown Income': 3, 'Understanding Society': 3, 'Govt Shutdown Crisis': 1}","{'Hurricane Katrina 2007': 13, 'COVID-19': 8, 'Govt Shutdown Income': 3, 'Understanding Society': 3, 'Govt Shutdown Crisis': 1}","{'Trauma / Health': 13, 'Employment': 7, 'Government Aid': 5, 'Housing / Shelter': 5, 'Economic / Income': 4, 'Financial Coping': 3, 'De...",2.079,5.667
